# 웹 크롤링 실습 (SeSAC, 2026.06.18)

requests + BeautifulSoup으로 정적 페이지 수집 연습.
강의 예제를 따라 해 본 뒤, 사이트 3곳에서 직접 데이터를 뽑아 본 과제.

| 미션 | 대상 | 수집 항목 | 결과 |
|---|---|---|---|
| 1 | quotes.toscrape.com | 명언 문장, 작성자 | 1페이지 10건 |
| 2 | 네이버 웹소설 통합랭킹 | 제목, 링크, 관심 수 | 15작품 |
| 3 | 랭킹 1위 작품 회차 목록 | 회차 제목, 링크, 평점 | 8페이지 72회차, CSV 저장 |

세 미션 모두 흐름은 같음
- 요청 → 상태 코드 확인 → HTML 파싱
- 반복되는 박스(한 건 = 태그 하나) 단위로 select
- 박스 안에서 필드 추출 → DataFrame

## 미션 1. 명언 사이트
- URL: https://quotes.toscrape.com/
- 목표: 명언 문장, 작성자
- 요청 헤더에 User-Agent 지정, timeout 5초

In [2]:
### 미션 1. 명언 사이트 크롤링

import pandas as pd  # 수집 결과를 표로 정리
import requests      # 페이지 요청
from bs4 import BeautifulSoup  # HTML에서 필요한 태그 찾기

In [4]:
user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36'
headers = {"User-Agent": user_agent}

target_url = "https://quotes.toscrape.com/"

response = requests.get(target_url, headers=headers, timeout=5)
print(response.status_code)  # 200인지 확인

200


In [5]:
soup = BeautifulSoup(response.text, "html.parser")  # HTML 파싱

In [16]:
quote_list = soup.select("div.quote")
print(quote_list)

[<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
<a class="tag" href="/tag/change/page/1/">change</a>
<a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>
<a class="tag" href="/tag/thinking/page/1/">thinking</a>
<a class="tag" href="/tag/world/page/1/">world</a>
</div>
</div>, <div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“It is our choices, Harry, that show what we truly are, far more than our abilities.”</span>
<span>by <small class="author" itemprop="author">J.K.

In [17]:
print(len(quote_list))

10


In [ ]:
data = []

for quote_box in quote_list:
	text = quote_box.select_one("span.text").text
	author = quote_box.select_one("small.author").text

	data.append({"명언문장" : text, "작성자" : author})

df = pd.DataFrame(data)

df

,명언문장,작성자
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
5,“Try not to become a man of success. Rather be...,Albert Einstein
6,“It is better to be hated for what you are tha...,André Gide
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin


### 메모
- 명언 1개 = `div.quote` 박스 1개 → 박스 단위로 돌면서 `span.text`, `small.author` 추출
- 1페이지만 수집(10건). 여러 페이지 순회는 미션 3에서 연습

## 미션 2. 네이버 웹소설 통합랭킹
- URL: https://novel.naver.com/webnovel/weekday
- 목표: 제목, 링크, 관심 수

In [ ]:
### 미션 2. 네이버 웹소설 통합랭킹 크롤링

'''
- 수집 URL: `https://novel.naver.com/webnovel/weekday`
- 수집 목표
	- 웹소설 제목
	- 웹소설 링크
	- 관심 수
'''
target_url = "https://novel.naver.com/webnovel/weekday"

response = requests.get(target_url, headers=headers, timeout=5)
print(response.status_code)  # 200인지 확인

200


In [22]:
# 웹소설 제목
soup = BeautifulSoup(response.text, "html.parser")  # HTML 파싱

web_novel_box = soup.select_one("div.component_section")
print(web_novel_box)


<div class="component_section">
<div class="component_head">
<h2 class="title">웹소설 통합 랭킹</h2>
<button class="total_set _closePreferGenreSettingAllLayer" id="integrationRankingSetBtn" type="button">통합설정</button>
<div class="ly_total_genre" id="integrationRankingSetDiv" style="display: none;">
<p class="tit">통합 랭킹 필터를 설정해보세요!</p>
<p class="dsc">장르와 리그를 선택하시면, 조건에 맞는 통합 랭킹 차트가 보여집니다.</p>
<ul class="genre_list" id="integrationRankingGenre">
<li>
<input class="blind" id="all.rom" type="checkbox" value="ROMANCE"/>
<label class="lb_genre" for="all.rom">로맨스</label>
</li>
<li>
<input class="blind" id="all.rof" type="checkbox" value="ROFAN"/>
<label class="lb_genre" for="all.rof">로판</label>
</li>
<li>
<input class="blind" id="all.fan" type="checkbox" value="FANTASY"/>
<label class="lb_genre" for="all.fan">판타지</label>
</li>
<li>
<input class="blind" id="all.mof" type="checkbox" value="MODERN_FANTASY"/>
<label class="lb_genre" for="all.mof">현판</label>
</li>
<li>
<input class="blind" id="all.hro" t

In [23]:
items = web_novel_box.select("li.item")
items

[<li class="item">
 <a class="link NPI=a:lis1" href="/best/list?novelId=1195394">
 <div class="thumbnail">
 <img alt="깨어난 야수" src="https://novel-phinf.pstatic.net/20250902_279/novel_1756786227259u1dfw_JPEG/sss.jpg?type=f100_80_2" width="100%"/>
 </div>
 <div class="info_area">
 <p class="ranking">
 <em class="rank">1</em>
 </p>
 <p class="title_group">
 <span class="bullet_wrap">
 <span class="bullet_up">UP</span>
 </span>
 <span class="title">깨어난 야수</span>
 </p>
 <p class="info_group">
 <span class="author">새벽별하나</span>
 <span class="genre">로맨스</span>
 </p>
 <p class="info_group">
 <span class="league">베스트리그</span>
 <span class="count">관심<span>1,810</span></span>
 </p>
 </div>
 </a>
 </li>,
 <li class="item">
 <a class="link NPI=a:lis2" href="/best/list?novelId=1222902">
 <div class="thumbnail">
 <img alt="살린 대가" src="https://novel-phinf.pstatic.net/20260414_282/novel_177614724188923UM4_PNG/ED919CECA780%2BEC9984EC84B12.png?type=f100_80_2" width="100%"/>
 </div>
 <div class="info_area"

In [ ]:
body = []

for item in items:
	title = item.select_one("span.title").text.strip()
	link = item.select_one("a.link")["href"]
	interest = item.select_one("p.info_group span.count").text.replace("관심", "").strip() 

	row = [title, link, interest]

	body.append([row[0],row[1],row[2]])

columns = ["웹소설 제목", "웹소설 링크", "관심 수"]
df = pd.DataFrame(body, columns=columns)

df

,웹소설 제목,웹소설 링크,관심 수
0,깨어난 야수,/best/list?novelId=1195394,"1,810"
1,살린 대가,/best/list?novelId=1222902,"1,453"
2,봄을 담다,/best/list?novelId=1137885,"2,017"
3,태오의 이유,/best/list?novelId=1223894,"1,166"
4,계약이 끝난 후에,/best/list?novelId=1225195,950
5,"착한 오빠, 나쁜 오빠",/best/list?novelId=1159312,"2,988"
6,교도소에서 나온 여자,/best/list?novelId=1211727,"1,778"
7,"언니, 이번 생엔 내가 왕비야",/webnovel/list?novelId=934373,2만
8,먹고 튄 여자.,/challenge/list?novelId=1228238,635
9,"로판 속 악역 황후입니다만,",/best/list?novelId=1212998,842


### 메모
- 15작품 수집
- 랭킹 영역(`div.component_section`)을 먼저 잡고 그 안에서 `li.item` 선택
- 관심 수가 '1,810' 같은 문자열 그대로 → 정렬·집계하려면 쉼표 제거 후 int 변환 필요
- 링크는 `/best/list?novelId=...` 형태의 상대 경로 → 미션 3 회차 목록 주소로 사용

## 미션 3. 랭킹 1위 작품 회차 목록
- 대상: 1위 「깨어난 야수」 (novelId=1195394)
- 목표: 회차 제목, 링크, 평점
- 순서: 1페이지 구조 확인 → 전체 페이지 순회 → 최종 결과만 CSV 저장

In [ ]:
 ### 미션 3. 특정 웹소설 회차 목록 크롤링
'''
- 수집 URL: 네이버 웹소설 통합랭킹 1위 작품의 상세/회차 목록 페이지
- 수집 목표
	- 회차 제목
	- 회차 링크
	- 평점
- 진행 목표
	- 먼저 1페이지의 회차 목록을 수집합니다.
	- 이후 전체 페이지를 순회하며 회차 목록을 수집합니다.
	- 미션 3의 최종 결과만 CSV 파일로 저장합니다.
	'''



In [ ]:

headers = {
	"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}



200


In [48]:
import time

# 데이터 보관용 바구니
novelbody = []

# 1~8페이지 반복
for page_num in range(1,9):
    # 페이지별 주소 생성
    target_url = f"https://novel.naver.com/best/list?novelId=1195394&page={page_num}"

    # 접속 성공할 때까지 재시도
    while True:
        response = requests.get(target_url, headers=headers, timeout=5)
        soup = BeautifulSoup(response.text, "html.parser")  # HTML 파싱

        # 회차 목록 전체 박스 추출
        novel_list_box = soup.select_one("ul.list_type2.v3.league_num")

        # 박스 찾으면 루프 탈출
        if novel_list_box:
            break
        
        # 실패 시 1초 대기
        time.sleep(1)

    # 개별 회차 데이터 추출
    for item in novel_list_box.select("li.volumeComment"):
            # 제목 ('UP' 지우고 공백 제거)
            title = item.select_one("p.subj").text.replace("UP", "").strip()
            # 링크
            link = item.select_one("a.list_item")["href"]
            # 평점 ('별점' 지우고 공백 제거)
            score = item.select_one("span.score_area").text.replace("별점", "").strip()
            # 바구니에 담기
            novelbody.append([title, link, score])

    # 페이지 전환 전 휴식
    time.sleep(0.3)
    
# 데이터프레임 변환
df = pd.DataFrame(novelbody, columns=["회차 제목", "회차 링크", "평점"])
# CSV 파일 저장 (한글 깨짐 방지)
df.to_csv("mission3_episodes.csv", index=False, encoding="utf-8-sig")
# 결과 출력
df

,회차 제목,회차 링크,평점
0,72. 철옹성(鐵甕城) 안으로,/best/detail?novelId=1195394&volumeNo=155,9.99
1,71. 두 개의 행복이 날아 들어왔다,/best/detail?novelId=1195394&volumeNo=154,9.98
2,"70. 새벽 3시, 멈춰버린 시계",/best/detail?novelId=1195394&volumeNo=153,10.0
3,69. 폭풍 전야의 태몽,/best/detail?novelId=1195394&volumeNo=152,10.0
4,"68. 지독한 양가감정, 안식과 갈증",/best/detail?novelId=1195394&volumeNo=151,9.96
...,...,...,...
67,5. 반지는 마음을 따라가는 법,/best/detail?novelId=1195394&volumeNo=87,10.0
68,4. 귀한 손님,/best/detail?novelId=1195394&volumeNo=86,9.98
69,3. 보고서에 핀 들국화,/best/detail?novelId=1195394&volumeNo=85,10.0
70,2. 반지가 주인을 만났습니다,/best/detail?novelId=1195394&volumeNo=84,9.98


### 결과
- 8페이지 순회, 72회차 수집 → `mission3_episodes.csv` 저장 (엑셀에서 한글 안 깨지게 `utf-8-sig`)

### 구현 포인트
- 회차 목록 박스(`ul.list_type2.v3.league_num`)가 안 잡히면 1초 쉬고 다시 요청
- 페이지 넘어갈 때마다 0.3초 대기
- 제목의 'UP', 평점의 '별점' 글자는 replace로 제거

### 다시 보니 고칠 점
- `while True` 재시도에 횟수 제한 없음 → 페이지 구조가 바뀌면 무한 루프. 최대 횟수를 두고 실패하면 건너뛰는 방식으로 바꿔야 함
- 페이지 범위를 1~8로 직접 지정 → 회차가 늘면 누락. 빈 목록이 나올 때 멈추는 방식이 더 안전
- 평점이 문자열 → 숫자로 분석하려면 float 변환 필요